### code for comparing quality of automatic and manual transcriptions

### imports

In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import quail
import pickle
import random
import re
import os
import json
import difflib
from num2words import num2words
from nltk.corpus import stopwords
from scipy.signal import resample
from scipy.stats import pearsonr, sem
from scipy.spatial.distance import cdist
from itertools import chain

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### set paths

In [2]:
data_dir = '../../../data/'
auto_dir = data_dir+'transcriptions/automatic/'
man_dir = data_dir+'transcriptions/manual/'

### load data

In [3]:
with open(data_dir+'pickles/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

In [4]:
def load_transcript(path):

    try:
        with open(path, 'r') as f:
            transcript = f.read()
        # format automatic transcripts
        if '.wav' in path:
            transcript = ' '.join([line.split(',')[0].lower() for line in transcript.split('\n')])
        else:
            transcript = transcript.lower() 
        return transcript

    except FileNotFoundError:
        print(f'No transcript at {path}')
        return
    

In [5]:
def get_confidence(sid, rectype):
    """
    Function to parse and compute average automatic transcriber 
    confidence rating for a single alternative, across all chunks 
    of a single decoded audio file
    
    Parameters
    ----------
    sid : str
        Across-session ID for the given participant
    rectype : str
        The current key while iteratively populating a participant's
        entry in `diffs`. Denotes the reference experimental session.
        One of: ['rec1', 'prediction', 'delayed', 'rec2']
        
    Returns
    ----------
    confidence : int
        mean confidence rating across the full decoded audio file
        
    """
    if rectype in ['rec1', 'prediction']:
        ses = 'session 1'
    else:
        ses = 'session 2'
    if 'rec' in rectype:
        ext = 'recall.wav.p'
    else:
        ext = f'{rectype}.wav.p'
    
    gcloud_path = os.path.join(auto_dir, sid, id_maps[sid][ses], f'{id_maps[sid][ses]}-{ext}')
    
    with open(gcloud_path, 'rb') as f:
        results_obj = pickle.load(f)
        
    return np.mean(list(chain(*[[res.alternatives[0].confidence for res in chunk.results] 
                                for chunk in results_obj])))


In [6]:
autos = dict.fromkeys(id_maps.keys())
mans = dict.fromkeys(id_maps.keys())

for sid, maps in id_maps.items():
    tid1 = maps['session 1']
    tid2 = maps['session 2']
    
    for method_dict, root in zip([autos, mans], [auto_dir, man_dir]):
        ext = '.txt'
        if 'automatic' in root:
            ext = f'-raw.wav{ext}'
            
        rec1_path = os.path.join(root, sid, tid1, f'{tid1}-recall{ext}')
        pred_path = os.path.join(root, sid, tid1, f'{tid1}-prediction{ext}')
        del_path = os.path.join(root, sid, tid2, f'{tid2}-delayed{ext}')
        rec2_path = os.path.join(root, sid, tid2, f'{tid2}-recall{ext}')

        zipit = zip(['rec1', 'prediction', 'delayed', 'rec2'], [rec1_path, pred_path, del_path, rec2_path])

        method_dict[sid] = {rectype : load_transcript(path) for rectype, path in zipit}

No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debugIEH2T:debugDLVLJ/debugIEH2T:debugDLVLJ-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debugIEH2T:debugDLVLJ/debugIEH2T:debugDLVLJ-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debug2Ea7T:debugosNZ7/debug2Ea7T:debugosNZ7-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debug2Ea7T:debugosNZ7/debug2Ea7T:debugosNZ7-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugBUnNA:debugLtZcs/debugBUnNA:debugLtZcs-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugBUnNA:debugLtZcs/debugBUnNA:debugLtZcs-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugQEynG:debugpwxCU/debugQEynG:debugpwxCU-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugQEynG:debugpwxCU/debugQEynG:debugpwxCU-recall.txt
No tra

No transcript at ../../../data/transcriptions/manual/MD-021519-A-01/debugzcmcY:debuguPcuX/debugzcmcY:debuguPcuX-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-021519-A-01/debugzcmcY:debuguPcuX/debugzcmcY:debuguPcuX-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-021519-A-01/debugpFb6R:debugVVofr/debugpFb6R:debugVVofr-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-021519-A-01/debugpFb6R:debugVVofr/debugpFb6R:debugVVofr-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-021819-B-01/debug2LQvu:debugk4zJe/debug2LQvu:debugk4zJe-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-021819-B-01/debug2LQvu:debugk4zJe/debug2LQvu:debugk4zJe-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-021819-B-01/debug5crvm:debug703dt/debug5crvm:debug703dt-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-021819-B-01/debug5crvm:debug703dt/debug5crvm:debug703dt-recall.txt
No tra

In [7]:
# check that both datasets contain all transcripts
for sid in id_maps.keys():
    if all(autos[sid].values()) and not all(mans[sid].values()):
        missing = ', '.join([k for k in mans[sid].keys() if mans[sid][k] is None])
        print(f'{sid} : no manual transcript for {missing}')
    if all(mans[sid].values()) and not all(autos[sid].values()):
        missing = ', '.join([k for k in autos[sid].keys() if autos[sid][k] is None])
        print(f'{sid} : no automatic transcript for {missing}')

MD-101218-A-01 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-B-01 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-A-02 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-B-02 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-A-03 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-B-03 : no manual transcript for rec1, prediction, delayed, rec2
MD-101218-A-04 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-B-01 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-B-02 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-A-02 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-A-03 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-B-03 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-A-04 : no manual transcript for rec1, prediction, delayed, rec2
MD-101318-B-04 : no manual transcript 

# compare transcript content directly

### compare lengths

In [8]:
# compare transcript lengths
print('Average number of words:')
print(f'automatic : {np.mean([len(autos[sid][k].split()) for k in autos[sid].keys() for sid in autos.keys()])}')
print(f'manual : {np.mean([len(mans[sid][k].split()) for k in mans[sid].keys() for sid in mans.keys() if mans[sid][k]])}')
print('\n')
print('Average number of (non-whitespace) characters:')
print(f'automatic : {np.mean([len(autos[sid][k].replace(" ","")) for k in autos[sid].keys() for sid in autos.keys()])}')
print(f'manual : {np.mean([len(mans[sid][k].replace(" ","")) for k in mans[sid].keys() for sid in mans.keys() if mans[sid][k]])}')


Average number of words:
automatic : 1232.0
manual : 1935.25


Average number of (non-whitespace) characters:
automatic : 4986.620689655172
manual : 7737.388888888889


## hit rate and false alarm rate

In [9]:
diffs = {sid : 
         {rectype : 
          None for rectype in ['rec1', 'prediction', 'delayed', 'rec2']
         } for sid in id_maps.keys()
        }

In [304]:
# for sid in diffs.keys():
#     for rectype in diffs[sid].keys():
        
#         # leave out participants whose manual transcriptions aren't finished for now
#         if not mans[sid][rectype]:
#             continue
            
#         man, auto = mans[sid][rectype].split(), autos[sid][rectype].split()
#         # generate deltas, trim leading fromfile and tofile tags
#         diff = list(difflib.unified_diff(auto, man, lineterm='', n=0))[2:]
#         # parse unified diff, get bounds of matching sequences
#         seqbounds = [idx for idx, word in enumerate(diff) if word.startswith('@') and idx!= 0] \
#         + [len(diff)]

#         misses = []
#         false_alarms = []
#         fa_quality = []
#         lbound = 1
#         for ubound in seqbounds:
#             seq = diff[lbound:ubound]
#             ms = [word[1:] for word in seq if word.startswith('+')]
#             fas = [word[1:] for word in seq if word.startswith('-')]
#             # track misses
#             misses.extend(ms)
#             if fas:
#                 # false alarms
#                 false_alarms.extend(fas)
#                 # track "quality" of false alarms
#                 for fa in fas:
#                     best_match = difflib.get_close_matches(fa, ms, n=1)
#                     if best_match:
#                         quality = difflib.SequenceMatcher(a=fa, b=best_match[0]).ratio()
#                         fa_quality.append((fa, best_match[0], quality))
#                     else:
#                         fa_quality.append((fa, None, 0))
            
#             lbound = ubound + 1
        
#         diffs[sid][rectype] = {
#             # prop words in auto that are not in man
#             'FAR' : len(false_alarms) / len(auto),
#             # prop words in auto that are also in man
#             'HR' : (len(auto) - len(false_alarms)) / len(man),
#             'fa_quality' : fa_quality,
#             'similarity' : difflib.SequenceMatcher(None, auto, man).ratio(),
#             'confidence' : get_confidence(sid, rectype)
#         }


## plot ROC curve

In [305]:
# hrs = []
# fars = []
# confs = []
# for rectype in diffs.values():
#     for data in rectype.values():
#         if data:
#             hrs.append(data['HR'])
#             fars.append(data['FAR'])
#             confs.append(data['confidence'])
            
# hrs = np.array(hrs)
# fars = np.array(fars)
# confs = np.array(confs)

In [306]:
# thresholds = np.arange(0, 1.1, .1)
# roc = {t : {} for t in thresholds}

# for i, thresh in enumerate(thresholds):
    
#     roc[thresh]['HR'] = hrs[np.where(np.digitize(confs, thresholds) == i)].mean()
#     roc[thresh]['FAR'] = fars[np.where(np.digitize(confs, thresholds) == i)].mean()
#     roc[thresh]['Hstd'] = hrs[np.where(np.digitize(confs, thresholds) == i)].std()
#     roc[thresh]['FAstd'] = fars[np.where(np.digitize(confs, thresholds) == i)].std()

/Users/paxtonfitzpatrick/anaconda/envs/py36/lib/python3.6/site-packages/ipykernel_launcher.py:6: RuntimeWarning: Mean of empty slice.
  
/Users/paxtonfitzpatrick/anaconda/envs/py36/lib/python3.6/site-packages/ipykernel_launcher.py:7: RuntimeWarning: Mean of empty slice.
  import sys
/Users/paxtonfitzpatrick/anaconda/envs/py36/lib/python3.6/site-packages/numpy/core/_methods.py:135: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)


In [10]:
# plt.plot([v['FAR'] for v in roc.values()], [v['HR'] for v in roc.values()])

In [11]:
# sns.regplot(fars, hrs)
# plt.ylabel('Hit Rate')
# plt.xlabel('False Alarm Rate')
# print(pearsonr(fars, hrs))

# different approach

In [12]:
def load_resp_obj(sid, rectype):
    """
    Function to load Google Cloud Speech response object for a
    decoded audio file
    
    Parameters
    ----------
    sid : str
        Across-session ID for the given participant
    rectype : str
        The current key while iteratively populating a participant's
        entry in `diffs`. Denotes the reference experimental session.
        One of: ['rec1', 'prediction', 'delayed', 'rec2']
        
    Returns
    ----------
    resp_obj : list of google.cloud.speech_v1.types.RecognizeResponse objects
        The  response objects for each chunk of a decoded audio file
        
    """
    if rectype in ['rec1', 'prediction']:
        ses = 'session 1'
    else:
        ses = 'session 2'
    if 'rec' in rectype:
        ext = 'recall.wav.p'
    else:
        ext = f'{rectype}.wav.p'
    
    gcloud_path = os.path.join(auto_dir, sid, id_maps[sid][ses], f'{id_maps[sid][ses]}-{ext}')
    
    with open(gcloud_path, 'rb') as f:
        resp_obj = pickle.load(f)
        
    return resp_obj

In [35]:
thresholds = np.arange(.1, 1.1, .1)
roc_data = {sid : 
            {t : 
             {} for t in thresholds
            } for sid in id_maps.keys()
           }

In [36]:
roc_data

{'MD-013119-A-02': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-013119-B-01': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-020119-A-02': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-020119-A-03': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-020119-A-04': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-020119-B-02': {0.1: {},
  0.2: {},
  0.30000000000000004: {},
  0.4: {},
  0.5: {},
  0.6: {},
  0.7000000000000001: {},
  0.8: {},
  0.9: {},
  1.0: {}},
 'MD-020819-B-01': {0.1: {},
  0.2: {},


In [26]:
for sid in roc_data.keys():
    for rectype in autos[sid].keys():
        # leave out participants whose manual transcriptions aren't finished for now
        if not mans[sid][rectype]:
            continue
        # load manual and automatic transcripts
        man, auto = mans[sid][rectype].split(), autos[sid][rectype].split()
        # load response object with confidences
        resp_obj = load_resp_obj(sid, rectype)
        # generate deltas, trim leading fromfile and tofile tags
        diff = list(difflib.unified_diff(auto, man, lineterm='', n=0))[2:]
        # parse unified diff, get bounds of matching sequences
        seqbounds = [idx for idx, word in enumerate(diff) if word.startswith('@') and idx!= 0] + [len(diff)]
        
        hits = []
        misses = []
        false_alarms = []
        fa_quality = []
        lbound = 1
        # index for list of response objects
        resp_ix = 0
        # index for results objects within a single response object
        result_ix = 0
        for ubound in seqbounds:
            seq = diff[lbound:ubound]
            print(seq)
            for word in seq:
                # remove +/-, look for word in current results object
                while word[1:] in resp_obj[resp_ix].results[result_ix]:
                    
            
            lbound = ubound + 1
        asdfasdfasdf

['-sell', '+so', '+the']
['-starch', '+starts']
['-well', '-and', '+al', '+and', '+we']
['-and', '-suites', '-on', '+al', '+and', '+basically', '+someone']
['-in', '-like', '-cakes', '+and', '+kicks']
['-with', '+which', '+is', '+like']
['-pushing', '-slot', '+and', '+earn', '+and', '+al', '+are', '+both', '+wearing', '+dark', '+colors', '+he', '+kicks', '+the', '+or', '+pushes', '+the', '+the']
['-on', '-brakes', '-and', '-is', '-very', '+and', '+breaks', '+it', '+and', '+al', '+is', '+really']
['-i', '-told', '+earn', '+tells']
['-the', '-ice', '-broke', '-the', '-mirror', '-and', '-then', '-on', '-also', '-around', '-after', '-talk', '-with']
['+then', '+earn', '+also', '+runs', '+after', '+al', '+because', '+he', "+doesn't", '+want', '+his', '+cousin', '+to', '+get', '+in', '+trouble', '+and', '+they', '+basically', '+talk', '+with', '+the', '+guy', '+who', '+broke', '+the', '+mirror', '+and']
['-paperboy', '+girl', "+he's", '+with', '+is', '+like', '+oh', "+you're", '+paper', '+bo

NameError: name 'asdfasdfasdf' is not defined

In [78]:
resp_obj[0].results[0].alternatives[0]

transcript: "sell episode starch in a car and there\'s a rapper or like a guy who the other person in the passenger seat refers to as well and don\'t have their names at the beginning of the episode but it\'s earn and his cousin and Suites on walks by in like cakes the window he\'s wearing a white shirt with contrasting pushing slot car side mirror on brakes and is very angry so he gets out of the car after I told him not to and he runs after the Ice broke the mirror and then on also around after"
confidence: 0.957231342792511
words {
  start_time {
    seconds: 1
    nanos: 800000000
  }
  end_time {
    seconds: 2
    nanos: 300000000
  }
  word: "sell"
}
words {
  start_time {
    seconds: 2
    nanos: 300000000
  }
  end_time {
    seconds: 3
    nanos: 200000000
  }
  word: "episode"
}
words {
  start_time {
    seconds: 3
    nanos: 200000000
  }
  end_time {
    seconds: 3
    nanos: 800000000
  }
  word: "starch"
}
words {
  start_time {
    seconds: 3
    nanos: 800000000
  }


In [73]:
autos[sid]['rec1']

'sell episode starch in a car and there\'s a rapper or like a guy who the other person in the passenger seat refers to as well and don\'t have their names at the beginning of the episode but it\'s earn and his cousin and suites on walks by in like cakes the window he\'s wearing a white shirt with contrasting pushing slot car side mirror on brakes and is very angry so he gets out of the car after i told him not to and he runs after the ice broke the mirror and then on also around after talk with the guy who broke the mirror and the girl he\'s with and the paperboy because he just made a new song called paperboy and then the screen flashes black and is that such scenes on and it flashes two a bird\'s-eye view of the city and it goes through wealthy affluent areas and then houses that are very nice and that\'s contrasted with the large houses with pools like well-manicured garden city and he see the words atlanta and you get earn in bed and he is listening to music on his block headphones

In [82]:
np.digitize(resp_obj[0].results[0].alternatives[0].confidence, roc_data[sid].keys())

ValueError: object of too small depth for desired array

[[0.9829309582710266,
  0.9370238184928894,
  0.9058111906051636,
  0.7585680484771729,
  0.8721573352813721,
  0.9493389129638672,
  0.7344788312911987,
  0.896754264831543,
  0.8275074362754822],
 [0.8816264867782593,
  0.8956366181373596,
  0.8948278427124023,
  0.9470463991165161,
  0.9401246309280396],
 [0.9680261015892029,
  0.8135889172554016,
  0.9050108194351196,
  0.8636961579322815,
  0.9608189463615417,
  0.9586723446846008,
  0.8988833427429199],
 [0.8943659067153931,
  0.9030069708824158,
  0.8198097944259644,
  0.8891328573226929,
  0.9419113397598267,
  0.9501504898071289,
  0.8859668374061584],
 [0.9154577851295471,
  0.8042872548103333,
  0.9290857911109924,
  0.9138996601104736],
 [0.9040009379386902,
  0.9832383990287781,
  0.914908230304718,
  0.9503338932991028,
  0.8676822781562805,
  0.8830572366714478,
  0.7884042263031006,
  0.9156187772750854],
 [0.9175805449485779,
  0.9165087938308716,
  0.9288980960845947,
  0.9569806456565857,
  0.9090355634689331,
  0.81

In [79]:
test = [7,5,6,8,0,2,3,4]

In [81]:
test

[7, 5, 6, 8, 2, 3, 4]

In [320]:
type(resp_obj[0])

google.cloud.speech_v1.types.RecognizeResponse

In [282]:
test

['@@ -1 +1,2 @@',
 '-showed',
 '+so',
 '+the',
 '@@ -7,2 +8 @@',
 '-he',
 '-was',
 "+who's",
 '@@ -16 +16 @@',
 '-shes',
 "+he's",
 '@@ -21,2 +21 @@',
 '-thing',
 '-is',
 '+like',
 '@@ -24 +23,3 @@',
 '-better',
 '+for',
 '+the',
 '+family',
 '@@ -26,3 +27,3 @@',
 '-like',
 '-for',
 '-basements',
 '+for',
 '+basically',
 '+his',
 '@@ -31 +32 @@',
 '-between',
 '+basically',
 '@@ -39,2 +40 @@',
 "-we're",
 '-praying',
 '+preparing',
 '@@ -43 +43 @@',
 '-book',
 '+boat',
 '@@ -49,2 +49 @@',
 '-going',
 '-to',
 '+gonna',
 '@@ -52 +51,2 @@',
 '-announcing',
 '+announced',
 '+as',
 '@@ -55,0 +56 @@',
 '+his',
 '@@ -58 +59 @@',
 '-and',
 '+but',
 '@@ -61 +61,0 @@',
 '-like',
 '@@ -66 +66 @@',
 '-is',
 "+there's",
 '@@ -68,2 +68 @@',
 '-there',
 '-is',
 "+there's",
 '@@ -72,2 +71,2 @@',
 '-jobe',
 '-two',
 '+gob',
 '+who',
 '@@ -86 +84,0 @@',
 '-like',
 '@@ -87,0 +86 @@',
 '+like',
 '@@ -92,2 +91,3 @@',
 '-the',
 '-secret',
 '+reveals',
 '+the',
 '+secrets',
 '@@ -94,0 +95 @@',
 '+a',
 '@@ -9

In [20]:
for sid in diffs.keys():
    for rectype in diffs[sid].keys():
        # leave out participants whose manual transcriptions aren't finished for now
        if not mans[sid][rectype]:
            continue
            
        man, auto = mans[sid][rectype].split(), autos[sid][rectype].split()
        # generate deltas, trim leading fromfile and tofile tags
        diff = list(difflib.unified_diff(auto, man, lineterm='', n=0))[2:]
        # parse unified diff, get bounds of matching sequences
        seqbounds = [idx for idx, word in enumerate(diff) if word.startswith('@') and idx!= 0] \
        + [len(diff)]

        misses = []
        false_alarms = []
        fa_quality = []
        lbound = 1
        for ubound in seqbounds:
            seq = diff[lbound:ubound]
            ms = [word[1:] for word in seq if word.startswith('+')]
            fas = [word[1:] for word in seq if word.startswith('-')]
            # track misses
            misses.extend(ms)
            if fas:
                # false alarms
                false_alarms.extend(fas)
                # track "quality" of false alarms
                for fa in fas:
                    best_match = difflib.get_close_matches(fa, ms, n=1)
                    if best_match:
                        quality = difflib.SequenceMatcher(a=fa, b=best_match[0]).ratio()
                        fa_quality.append((fa, best_match[0], quality))
                    else:
                        fa_quality.append((fa, None, 0))
            
            lbound = ubound + 1
        
        diffs[sid][rectype] = {
            # prop words in auto that are not in man
            'FAR' : len(false_alarms) / len(auto),
            # prop words in auto that are also in man
            'HR' : (len(auto) - len(false_alarms)) / len(man),
            'fa_quality' : fa_quality,
            'similarity' : difflib.SequenceMatcher(None, auto, man).ratio(),
            'confidence' : get_confidence(sid, rectype)
        }


In [21]:
seq

['-and',
 '+he',
 '+realized',
 '+that',
 '+like',
 '+most',
 '+of',
 '+them',
 '+work',
 '+in',
 '+a',
 '+little',
 '+local',
 '+theatre',
 '+and',
 "+he's",
 '+like',
 '+i',
 '+wanna',
 '+be',
 '+an',
 '+actor',
 '+and',
 '+so',
 '+he',
 '+starts',
 '+like',
 '+auditioning',
 '+for',
 '+acting',
 '+positions',
 '+and',
 "+that's",
 '+i',
 '+think',
 '+the',
 '+episode']